# 00 - Chuẩn bị tập dữ liệu benchmark (GSO 29 vật thể)

**Mục đích**: tạo ảnh đầu vào (adapter) riêng cho từng mô hình từ 29 ảnh đã được chuẩn hóa:

- `adapters/triposr/`     : RGBA → blend nền xám 0.5 → **RGB** (đúng input TripoSR của C1)
- `adapters/instantmesh/`: RGBA giữ nguyên (InstantMesh dùng alpha, chạy `--no_rembg`)
- `adapters/hunyuan3d2mini/`: RGBA giữ nguyên (Hunyuan3D-2mini dùng alpha)

**Quan trọng**: việc tách nền (rembg u2net) + cắt + pad 0.85 + resize 512 đã được thực hiện sẵn
bằng **đúng phiên bản rembg 2.0.52 của nghiên cứu C1** trên máy nguồn — Colab **không cần rembg**
(Colab dùng Python 3.12+ nên không cài được rembg 2.0.52). Kết quả đã được xác minh trùng pixel
100% với input C1 (29/29 PASS) — notebook này chỉ kiểm tra lại bằng SHA-256.

**Thứ tự dùng**: notebook này → 01 → (duyệt smoke) → 02 → 03 → 05.

**Quy tắc bắt buộc**: không sửa code; cell nào lỗi → chụp màn hình gửi về, không tự sửa.

## PHẦN 1 - Kết nối Google Drive

Chạy cell dưới, nhấn link, đăng nhập Google, copy mã xác nhận dán vào ô rồi Enter.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

DRIVE_ROOT = '/content/drive/MyDrive/nien_luan_3d_benchmark'
PACKAGE_DIR = os.path.join(DRIVE_ROOT, 'dataset', 'package')
DATASET_DIR = os.path.join(DRIVE_ROOT, 'dataset')

os.makedirs(PACKAGE_DIR, exist_ok=True)

zip_path = os.path.join(PACKAGE_DIR, 'gso29_standardized.zip')
std_manifest = os.path.join(PACKAGE_DIR, 'standardized_manifest.csv')

print('zip ok    :', os.path.exists(zip_path), zip_path)
print('manifest ok:', os.path.exists(std_manifest), std_manifest)
assert os.path.exists(zip_path), 'THIEU gso29_standardized.zip - chua upload 2 file vao Google Drive!'
assert os.path.exists(std_manifest), 'THIEU standardized_manifest.csv - chua upload 2 file vao Google Drive!'

### Giải nén + xác minh SHA-256 (theo PIXEL)

Mỗi ảnh chuẩn hóa phải có SHA-256 **của pixel RGBA** khớp với `standardized_manifest.csv` (column `standardized_rgba_sha256` — hash pixel nguyên gốc của C1, đã được tạo trên máy nguồn bằng rembg 2.0.52). So sánh theo pixel nên không phụ thuộc metadata của file PNG. Có ảnh FAIL → **dừng lại, chụp màn hình, báo về**.

In [ ]:
import csv, hashlib, zipfile

import numpy as np
from PIL import Image

def sha256_pixels(pil_img):
    return hashlib.sha256(np.asarray(pil_img).tobytes()).hexdigest().upper()

STD_DIR = os.path.join(DATASET_DIR, 'standardized_rgba')
if not os.path.exists(STD_DIR) or len(os.listdir(STD_DIR)) < 29:
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(DATASET_DIR)
    print('Giai nen xong:', STD_DIR)

exp = {}
with open(std_manifest, encoding='utf-8-sig') as f:
    for r in csv.DictReader(f):
        exp[r['object_id']] = r

fails = []
for oid, meta in sorted(exp.items()):
    p = os.path.join(STD_DIR, oid + '.png')
    if not os.path.exists(p):
        fails.append((oid, 'THIEU FILE'))
    else:
        img = Image.open(p).convert('RGBA')
        if sha256_pixels(img) != meta['standardized_rgba_sha256']:
            fails.append((oid, 'SHA-256 KHAC'))

print('Tong so anh:', len(exp))
print('PASS:', len(exp) - len(fails))
if fails:
    for oid, why in fails:
        print('FAIL:', oid, why)
    raise SystemExit('Co anh FAIL - DUNG LAI va bao ve.')
print('TAT CA PASS. Sang phan 2.')

## PHẦN 2 - Tạo ảnh adapter cho 3 mô hình

Từ ảnh RGBA chuẩn hóa 512×512, tạo:
- `adapters/triposr/`       : blend nền xám 0.5 → RGB (công thức y hệt `eval_common.preprocess_image`)
- `adapters/instantmesh/`   : RGBA (không đổi)
- `adapters/hunyuan3d2mini/`: RGBA (không đổi)

Ghi `dataset_manifest.csv` (SHA-256 từng loại) — các notebook 01/02/03 sẽ đọc file này.

In [ ]:
import numpy as np
from PIL import Image

GRAY_BLEND = 0.5

def sha256_pixels(pil_img):
    return hashlib.sha256(np.asarray(pil_img).tobytes()).hexdigest().upper()

def adapter_triposr(img_rgba):
    arr = np.asarray(img_rgba).astype(np.float32) / 255.0
    rgb = arr[:, :, :3]
    a = arr[:, :, 3:4]
    out = rgb * a + (1.0 - a) * GRAY_BLEND
    return Image.fromarray((np.clip(out, 0, 1) * 255.0).astype(np.uint8))

ADAPTERS = {
    'triposr': os.path.join(DATASET_DIR, 'adapters', 'triposr'),
    'instantmesh': os.path.join(DATASET_DIR, 'adapters', 'instantmesh'),
    'hunyuan3d2mini': os.path.join(DATASET_DIR, 'adapters', 'hunyuan3d2mini'),
}
for d in list(ADAPTERS.values()):
    os.makedirs(d, exist_ok=True)

fields = ['object_id', 'object_name', 'category', 'gt_up_axis', 'eligible_quality',
          'quality_exclusion_reason', 'source_sha256', 'source_bytes',
          'standardized_rgba_sha256', 'adapter_triposr_sha256',
          'adapter_instantmesh_sha256', 'adapter_hunyuan3d2mini_sha256',
          'standard_input_hash_match', 'status', 'error']
rows = []

for oid, meta in sorted(exp.items()):
    row = {k: meta.get(k, '') for k in fields[:10]}
    try:
        std = Image.open(os.path.join(STD_DIR, oid + '.png')).convert('RGBA')
        for name, outdir in ADAPTERS.items():
            img = adapter_triposr(std) if name == 'triposr' else std
            img.save(os.path.join(outdir, oid + '.png'))
            row['adapter_%s_sha256' % name] = sha256_pixels(img)
        row['status'] = 'success'
    except Exception as e:
        row['status'] = 'failed'
        row['error'] = '%s: %s' % (type(e).__name__, e)
    rows.append(row)

with open(os.path.join(DATASET_DIR, 'dataset_manifest.csv'), 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(rows)

ok = sum(1 for r in rows if r['status'] == 'success')
print('Tao adapter xong: %d/%d success' % (ok, len(rows)))
for r in rows:
    if r['status'] != 'success':
        print('FAIL:', r['object_id'], r['error'])
if ok != len(rows):
    raise SystemExit('Co loi - DUNG LAI va bao ve.')

## PHẦN 3 - Xác minh adapter TripoSR trùng với input C1

So sánh SHA-256 (pixel) của `adapters/triposr/<id>.png` với giá trị đã xác minh trên máy nguồn
(chính là `input_hash` trong log ablation C1).
**Tất cả PASS** → đầu vào benchmark này giống hệt đầu vào nghiên cứu C1.

In [ ]:
matches = 0
for r in rows:
    if r['status'] != 'success':
        r['standard_input_hash_match'] = 'ERROR'
        continue
    expected = r['adapter_triposr_sha256'].lower()
    got = sha256_pixels(Image.open(os.path.join(ADAPTERS['triposr'], r['object_id'] + '.png'))).lower()
    r['standard_input_hash_match'] = 'PASS' if got == expected else 'FAIL'
    if r['standard_input_hash_match'] == 'PASS':
        matches += 1

with open(os.path.join(DATASET_DIR, 'dataset_manifest.csv'), 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(rows)

print('standard_input_hash_match: PASS=%d / %d' % (matches, len(rows)))
for r in rows:
    if r['standard_input_hash_match'] != 'PASS':
        print('  %-12s %s %s' % (r['object_id'], r['standard_input_hash_match'], r.get('error', '')))

## KẾT THÚC notebook 00

✅ Hoàn thành khi:
1. `dataset_manifest.csv` có 29 dòng, tất cả `status=success`;
2. `standard_input_hash_match` toàn PASS.

**Bước tiếp theo**: mở `01_triposr_t4_benchmark.ipynb`. Trước khi chạy batch, chạy PHẦN 4 (smoke test) và **dừng lại để chờ duyệt**.